# Zonas de vida de Holdridge (38 zonas) — CHELSA V2.1, normal 1991-2020

Versão em Python (API do Earth Engine) do script JS anterior, adaptada aos **três assets separados por
variável** (`tas`, `pet`, `pr`; 12 bandas mensais cada), gerados por `gerar_normal_multibanda.py`.

**O que mudou em relação ao script antigo:** os assets já estão em unidades físicas
(tas em °C; pet e pr em mm/mês). Por isso **não há mais** `subtract(273.15)`, `divide(100)` nem `divide(10)`.
A biotemperatura, a classificação por classes e a tabela das 38 zonas seguem a lógica do script original.
O cálculo está em [holdridge_gee.py](holdridge_gee.py).

A suavização por moda (focalMode) do script JS foi trazida de volta (seção 3b): `brasil` (contorno real do
país, FAO GAUL) é usado para `clip`/geometria/mapa e também para recortar `zonas_finais` (uma por versão de ETP) de volta ao contorno
do país depois da suavização (que pode espalhar valores um pouco além da borda). `zonas_finais` (uma por versão de ETP) tem as mesmas
características do que seria exportado manualmente no GEE. Este notebook **não exporta** nada
automaticamente; a exportação fica a cargo do usuário.

**Numeração e nomes das 38 zonas:** Jungkunst et al. (2021, J. Plant Nutr. Soil Sci. 184:5-11, Tab. 1 e
Fig. 1), base Leemans (1990) -- legenda em `h.LEGENDA`. A tabela de zonas do script JS original usava em
todas as faixas térmicas o padrão do subtropical, o que deslocava a faixa tropical em uma zona (ETP/P 0,5-1
virava 36, "Tropical dry forest", em vez de 37, "Tropical moist forest") e também o temperado frio, o boreal
e o subpolar. Corrigido em [holdridge_gee.py](holdridge_gee.py) (2026-09-24); temperado quente e
subtropical já estavam certos.

**Duas versões de ETP na razão ETP/P** (testadas lado a lado na análise solo-clima, `corelacao/`):
- **ETPM:** ETP de Penman-Monteith do CHELSA (`pet`).
- **ETH:** ETP de Holdridge, 58,93 x biotemperatura -- a definição original, usada por Leemans e por
  Jungkunst et al. (2021).

Com a ETH ~23% dos pixels do Brasil mudam de zona (o país fica mais úmido, sobretudo no Sul e Sudeste).
A biotemperatura, e portanto a faixa térmica, é a mesma nas duas.

## 1. Configuração

In [1]:
import sys
sys.path.insert(0, ".")

import ee
import geemap
import pandas as pd
import holdridge_gee as h

PROJETO = "fcoliveira"

# Assets separados por variavel (12 bandas mensais cada; ajuste para o caminho onde voce subiu as imagens)
ASSET_TAS = "projects/fcoliveira/assets/chelsa_brasil_tas_normal_1991_2020"
ASSET_PET = "projects/fcoliveira/assets/chelsa_brasil_pet_normal_1991_2020"
ASSET_PR = "projects/fcoliveira/assets/chelsa_brasil_pr_normal_1991_2020"

# Duas versoes da ETP na razao ETP/P: Penman-Monteith do CHELSA e a de Holdridge (58,93 x biotemperatura)
VERSOES_ETP = {"ETPM": "penman", "ETH": "holdridge"}

# Assets de saida (referencia; a exportacao e manual, ver secao 6)
ASSET_SAIDA = {v: f"projects/fcoliveira/assets/Holdridge_CHELSA_BR_1991_2020_{v}" for v in VERSOES_ETP}

# 24  = correcao de latitude so nos meses com t > 24 C (formulacao original de Holdridge).
# None = correcao em todos os meses (como o script JS antigo); zera os meses frios no Sul
#        e gera zonas de gelo/polar espurias. Veja a secao 4.
LIMIAR_CORRECAO = 24


## 2. Earth Engine e região (Brasil)

In [2]:
try:
    ee.Initialize(project=PROJETO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO)

# 'brasil' usa o contorno real do pais (FAO GAUL) para clip/geometria/mapa.
brasil = (ee.FeatureCollection("FAO/GAUL/2015/level0")
          .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))
estados = (ee.FeatureCollection("FAO/GAUL/2015/level1")
           .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
/Users/coted-sh/Python/MapBiomas_analises/climas/chelsa_climas_brasil/.venv/lib/python3.13/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


## 3. Carregar a normal e classificar

In [3]:
normal = h.carregar_normal(ASSET_TAS, ASSET_PET, ASSET_PR)
print("Bandas:", normal.bandNames().getInfo())

resultados = {v: h.classificar_holdridge(normal, LIMIAR_CORRECAO, etp=e).clip(brasil)
              for v, e in VERSOES_ETP.items()}
zonas = {v: r.select("zone38_id") for v, r in resultados.items()}

Bandas: ['tas_01', 'tas_02', 'tas_03', 'tas_04', 'tas_05', 'tas_06', 'tas_07', 'tas_08', 'tas_09', 'tas_10', 'tas_11', 'tas_12', 'pet_01', 'pet_02', 'pet_03', 'pet_04', 'pet_05', 'pet_06', 'pet_07', 'pet_08', 'pet_09', 'pet_10', 'pet_11', 'pet_12', 'pr_01', 'pr_02', 'pr_03', 'pr_04', 'pr_05', 'pr_06', 'pr_07', 'pr_08', 'pr_09', 'pr_10', 'pr_11', 'pr_12']


## 3b. Suavização (focalMode) e recorte final

Reproduz as etapas finais do script JS original: filtro de moda 3x3 na zona classificada e recorte de
volta ao contorno de `brasil` (a suavização pode espalhar valores por 1 pixel além da borda; o `clip`
final remove essa sobra e também define como "sem zona" (0) os pixels dentro do país que não receberam
classificação). O resultado, `zonas_finais` (uma por versão de ETP), é a imagem pronta para a exportação manual (seção 6).

In [4]:
zonas_finais = {v: (h.suavizar_zona(z).unmask(0).clip(brasil)
                    .rename("zone38_id")
                    .toByte())
                for v, z in zonas.items()}

## 4. Verificação: áreas por zona

No Brasil não se espera gelo/polar (zonas 1 e 2). A correção de latitude `t - 0,03 * lat * (t - 24)^2` é, na
formulação original de Holdridge, aplicada só aos meses com t > 24 °C (`LIMIAR_CORRECAO = 24`). **Se aplicada em
todos os meses** (`LIMIAR_CORRECAO = None`, como no script JS antigo), o termo quadrático também desconta dos meses
frios e, com temperaturas em °C corretas, derruba o inverno a zero em latitudes maiores que ~25°, gerando zonas
1-2 espúrias no Sul. Se elas aparecerem, confira `LIMIAR_CORRECAO`.

In [5]:
def area_por_zona(zona_final):
    hist = zona_final.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=brasil.geometry(),
        scale=5000,
        maxPixels=1e13,
        bestEffort=True,
    ).get("zone38_id").getInfo()
    s = pd.Series(hist, dtype=float)
    s.index = s.index.astype(float).astype(int)
    return 100 * s / s.sum()

tabela = pd.DataFrame({v: area_por_zona(z) for v, z in zonas_finais.items()}).fillna(0).sort_index()
tabela.insert(0, "nome", tabela.index.map(h.LEGENDA).fillna("sem zona"))
tabela.index.name = "zona"
display(tabela.round(2).rename(columns={v: f"% área {v}" for v in VERSOES_ETP}))

for v in VERSOES_ETP:
    if tabela.loc[tabela.index.isin([1, 2]), v].sum() > 0:
        print(f"ATENCAO ({v}): parte do Brasil caiu nas zonas 1-2 (gelo/polar). Reveja LIMIAR_CORRECAO.")

,nome,% área ETPM,% área ETH
zona,,,
0,sem zona,0.00,0.00
16,Cool temperate wet forest,0.00,0.00
21,Warm temperate dry forest,0.07,0.00
22,Warm temperate moist forest,1.86,1.40
23,Warm temperate wet forest,0.20,0.72
24,Warm temperate rain forest,0.00,0.00
26,Subtropical desert bush,0.28,0.00
27,Subtropical thorn steppe,4.50,2.36
28,Subtropical dry forest,24.15,14.31


## 5. Mapa

In [6]:
vis = {"min": 1, "max": 38, "palette": h.PALETA}

Map = geemap.Map(center=[-14, -52], zoom=4)
for v, z in zonas_finais.items():
    Map.addLayer(z.selfMask(), vis, f"Holdridge {v} (38 zonas)", v == "ETPM")
r = resultados["ETPM"]
Map.addLayer(r.select("biotemp"), {"min": 10, "max": 30, "palette": ["blue", "yellow", "red"]}, "Biotemperatura", False)
Map.addLayer(r.select("prec"), {"min": 0, "max": 3500, "palette": ["white", "blue"]}, "Precipitacao anual", False)
for v, rv in resultados.items():
    Map.addLayer(rv.select("retp"), {"min": 0, "max": 4, "palette": ["green", "yellow", "red"]}, f"Razao ETP/P {v}", False)
Map.addLayer(estados.style(color="000000", fillColor="00000000", width=1), {}, "Estados")
Map.addLayer(brasil.style(color="000000", fillColor="00000000", width=2), {}, "Brasil")
Map.add_colorbar(vis, label="Zona de Holdridge (id 1-38)", layer_name="Holdridge ETPM (38 zonas)")
Map

Map(center=[-14, -52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

## 6. Exportação

A célula abaixo baixa as duas `zonas_finais` (ETPM e ETH) como GeoTIFFs locais, na resolução nativa do CHELSA (~928 m, a mesma
dos assets de entrada) e recortado por `brasil`. O arquivo sai em `climas/dados_chelsa/holdridge/`, no
mesmo padrão do que `gerar_normal_multibanda.py` já faz para tas/pet/pr: gera local e depois sobe manualmente
para o GEE (via bucket do GCS, igual ao fluxo em `chelsa_brasil.ipynb`).

In [7]:
from pathlib import Path

SAIDA_DIR = Path("../dados_chelsa/holdridge")
SAIDA_DIR.mkdir(parents=True, exist_ok=True)
proj = normal.select(0).projection()

for v, z in zonas_finais.items():
    nome = f"Holdridge_CHELSA_BR_1991_2020_{v}.tif"
    caminho_local = SAIDA_DIR / nome
    # download_ee_image (nao ee_export_image) porque a imagem passa do limite de 48MB de
    # download direto do GEE; ele baixa em tiles e remonta um unico GeoTIFF.
    geemap.download_ee_image(
        z,
        filename=str(caminho_local),
        region=brasil.geometry(),
        crs=proj.crs().getInfo(),
        scale=proj.nominalScale().getInfo(),
        dtype="uint8",
    )
    print(f"Salvo em: {caminho_local.resolve()}")
    print(f"  gsutil cp {caminho_local} gs://SEU_BUCKET/{nome}")
    print(f"  earthengine upload image --asset_id={ASSET_SAIDA[v]} --pyramiding_policy=mode gs://SEU_BUCKET/{nome}")

  0%|          |0/20 tiles [00:00<?]

Salvo em: /Users/coted-sh/Python/MapBiomas_analises/climas/dados_chelsa/holdridge/Holdridge_CHELSA_BR_1991_2020_ETPM.tif
  gsutil cp ../dados_chelsa/holdridge/Holdridge_CHELSA_BR_1991_2020_ETPM.tif gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020_ETPM.tif
  earthengine upload image --asset_id=projects/fcoliveira/assets/Holdridge_CHELSA_BR_1991_2020_ETPM --pyramiding_policy=mode gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020_ETPM.tif


  0%|          |0/20 tiles [00:00<?]

Salvo em: /Users/coted-sh/Python/MapBiomas_analises/climas/dados_chelsa/holdridge/Holdridge_CHELSA_BR_1991_2020_ETH.tif
  gsutil cp ../dados_chelsa/holdridge/Holdridge_CHELSA_BR_1991_2020_ETH.tif gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020_ETH.tif
  earthengine upload image --asset_id=projects/fcoliveira/assets/Holdridge_CHELSA_BR_1991_2020_ETH --pyramiding_policy=mode gs://SEU_BUCKET/Holdridge_CHELSA_BR_1991_2020_ETH.tif
